In [36]:
import cv2, glob, os

import numpy as np

import mediapipe as mp
from mediapipe.tasks.python import vision

import pandas as pd
import matplotlib.pyplot as plt

In [37]:
def extract_video_features(video_path, step=30):
    """
    Extraherar blendshapes från en video med ett givet steg och
    returnerar en aggregerad vektor (mean, max, std) samt rå-dataframe.
    """
    BaseOptions = mp.tasks.BaseOptions
    FaceLandmarker = vision.FaceLandmarker
    FaceLandmarkerOptions = vision.FaceLandmarkerOptions
    VisionRunningMode = vision.RunningMode

    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="face_landmarker.task"),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=1,
        output_face_blendshapes=True,
    )

    results_list = []
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total_steps = total_frames // step

    frame_idx = 0
    n = 0
    with FaceLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened() and n < total_steps:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx > total_frames:
                raise IndexError()

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            result = landmarker.detect(mp_image)

            if result.face_blendshapes:
                blendshapes = result.face_blendshapes[0]
                bs_dict = {
                    category.category_name: category.score for category in blendshapes
                }
                bs_dict["frame_idx"] = frame_idx
                results_list.append(bs_dict)

            frame_idx += step
            n += 1

    cap.release()

    if not results_list:
        return None, None

    df_video = pd.DataFrame(results_list)

    # Plocka ut enbart blendshape-kolumnerna (exkludera frame_idx)
    bs_cols = [c for c in df_video.columns if c != "frame_idx"]

    # Beräkna mean, max och std över alla samplade frames för denna video
    feat_mean = df_video[bs_cols].mean().values
    feat_max = df_video[bs_cols].max().values
    feat_std = df_video[bs_cols].std().values

    # Concateneta till en gemensam vektor
    feature_vector = np.concatenate([feat_mean, feat_max, feat_std])

    return feature_vector, df_video

In [38]:
def parse_annotation_file(txt_path):
    """Läser din train/test.txt-fil rad för rad."""
    data = []
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            
            video_name = parts[0]
            labels = [int(x) for x in parts[1:7]]
            gender = parts[7]
            score = float(parts[8])
            
            data.append({
                'video_name': video_name,
                'labels': labels, # Sparar alla 6 som en lista
                'gender': gender,
                'score': score
            })
    return pd.DataFrame(data)


In [39]:
# --- Konfiguration ---
video_folder = r"data\ElderReact_Data\ElderReact_train"      # Byt till din sökväg
txt_path = r"ElderReact\Annotations\train_labels.txt"            # Byt till din txt-fil
step_size = 30                            # Samplingssteg

# Läs in annotationer
df_ann = parse_annotation_file(txt_path)

X_features = []
y_labels = []
y_scores = []
valid_videos = []

print(" खेलोंar att bearbeta videor...")

max_count = 10
count = 0
# Gå igenom filen rad för rad
for index, row in df_ann.iterrows():
    if count >= max_count:
        break 

    video_name = row['video_name']
    video_path = os.path.join(video_folder, video_name)
    
    if not os.path.exists(video_path):
        print(f"Hittade inte videon: {video_name}, hoppar över.")
        continue
        
    print(f"Bearbetar: {video_name}...")
    
    # 1. Extrahera den concaterade feature-vektorn (mean + max + std)
    feat_vec, _ = extract_video_features(video_path, step=step_size)
    
    if feat_vec is not None:
        X_features.append(feat_vec)
        y_labels.append(row['labels'])
        y_scores.append(row['score'])
        valid_videos.append(video_name)

    count +=1

# Gör om till NumPy-matriser redo för ML/klustring
X = np.array(X_features)
y = np.array(y_labels)
scores = np.array(y_scores)

print("\n--- Klart! ---")
print(f"Lyckades extrahera funktioner för {len(X)} av {len(df_ann)} videor.")
print(f"X shape (features): {X.shape}")
print(f"y shape (labels):   {y.shape}")

 खेलोंar att bearbeta videor...
Bearbetar: 50_50_100.mp4...
Bearbetar: 50_50_104.mp4...
Bearbetar: 50_50_105.mp4...
Bearbetar: 50_50_106.mp4...
Bearbetar: 50_50_22.mp4...
Bearbetar: 50_50_24.mp4...
Bearbetar: 50_50_27.mp4...
Bearbetar: 50_50_4.mp4...
Bearbetar: 50_50_45.mp4...
Bearbetar: 50_50_46.mp4...

--- Klart! ---
Lyckades extrahera funktioner för 10 av 615 videor.
X shape (features): (10, 156)
y shape (labels):   (10, 6)


In [41]:
print(X_features[1][:])

[3.93178544e-06 2.71139462e-01 6.45087406e-01 1.74034561e-05
 3.43335546e-02 1.13973563e-03 7.08786720e-04 1.12680810e-07
 7.40492210e-08 4.43285502e-01 7.03793615e-01 6.01220950e-01
 7.43328720e-01 2.32679938e-02 3.60855039e-01 3.86794530e-01
 2.39667320e-02 2.38613950e-02 2.67631232e-03 7.03478128e-01
 5.65777332e-01 4.11347119e-03 1.38700540e-04 6.41897170e-04
 1.65450640e-03 2.57481255e-02 1.19721976e-03 1.26687382e-03
 1.39588362e-01 3.86239212e-01 1.18787350e-04 9.09722130e-05
 2.93462615e-03 9.91705365e-05 1.62765147e-02 1.28589493e-02
 1.86335521e-01 3.05987459e-01 6.30982066e-04 4.06320917e-02
 7.83679639e-02 1.55429462e-01 4.65539988e-02 4.67833350e-02
 3.35679313e-01 5.01652971e-01 3.40867550e-03 1.35252519e-03
 4.29223972e-02 1.96526609e-01 4.65498477e-07 4.57129183e-07
 5.44344039e-06 4.26617771e-01 6.82796180e-01 2.76095780e-05
 4.02544439e-02 1.80055853e-03 1.19336869e-03 1.69603609e-07
 9.32568511e-08 5.47956884e-01 8.03540826e-01 7.49464035e-01
 8.22690964e-01 4.597171